# 01 — The Artificial Neuron, Visually

> **Goal:** understand a neuron as a geometric transformation — not as a black-box formula.

An artificial neuron takes inputs, scales them with **weights**, adds a **bias**, and optionally passes the result through an **activation function**.

### The entire neuron in two equations

$$
z = w_1x_1 + w_2x_2 + \cdots + w_nx_n + b
$$

$$
a = \phi(z)
$$

| Symbol | Meaning | Intuition |
|---|---|---|
| $x_i$ | input | information entering the neuron |
| $w_i$ | weight | how strongly an input matters |
| $b$ | bias | shifts the response |
| $z$ | pre-activation | weighted evidence before nonlinearity |
| $\phi$ | activation | nonlinear transformation |
| $a$ | output | value passed to the next layer |

> The important question throughout this notebook is: **what changes geometrically when a parameter changes?**

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from IPython.display import display, HTML
import ipywidgets as widgets

np.set_printoptions(precision=4, suppress=True)

## 1. A neuron as a weighted sum

For two inputs, the pre-activation is:

$$z = w_1x_1 + w_2x_2 + b$$

We will compute every term explicitly before introducing any abstraction.

In [ ]:
x = np.array([2.0, -1.0])
w = np.array([0.8, -1.2])
b = 0.5

z = x @ w + b
print("input x =", x)
print("weights w =", w)
print("bias b =", b)
print("weighted sum z =", z)

### Read the parameters geometrically

- **Positive weight:** increasing that input pushes $z$ upward.
- **Negative weight:** increasing that input pushes $z$ downward.
- **Large $|w_i|$:** the neuron is more sensitive to that input.
- **Bias $b$:** shifts the response even when the inputs stay unchanged.

> A weight controls **sensitivity**. A bias controls **offset**.

## 2. One-input neuron: live geometry

With one input, the neuron before activation is simply:

$$z = wx + b$$

That is a line. This gives us the cleanest possible visual interpretation:

- $w$ changes the **slope**.
- $b$ changes the **vertical position**.

Change them below and watch the same neuron become a different transformation.

In [ ]:
def neuron_1d(x, w, b):
    return w*x + b

x_grid = np.linspace(-5, 5, 300)

w_slider = widgets.FloatSlider(value=1.0, min=-3, max=3, step=0.1, description='weight')
b_slider = widgets.FloatSlider(value=0.0, min=-5, max=5, step=0.1, description='bias')

def update_line(w, b):
    y = neuron_1d(x_grid, w, b)
    plt.figure(figsize=(9,5))
    plt.axhline(0, linewidth=1)
    plt.axvline(0, linewidth=1)
    plt.plot(x_grid, y)
    plt.scatter([0], [b], s=80, label=f'intercept = bias = {b:.2f}')
    plt.title(f'y = {w:.2f}x + {b:.2f}')
    plt.xlabel('input x')
    plt.ylabel('pre-activation z')
    plt.ylim(-10,10)
    plt.grid(alpha=.25)
    plt.legend()
    plt.show()

widgets.interact(update_line, w=w_slider, b=b_slider);

### What should move in your head as you move the controls?

| Change | Visual effect | Neural interpretation |
|---|---|---|
| increase $w$ | line becomes steeper | stronger positive sensitivity |
| make $w$ negative | line flips direction | input contribution reverses |
| increase $b$ | line moves upward | activation threshold shifts |
| set $w=0$ | flat line | neuron ignores the input |

A neuron is therefore already a tiny **learnable geometric transformer**.

## 3. Activation functions

The weighted sum alone is linear. The activation function introduces nonlinearity:

$$a = \phi(z)$$

Three common choices are:

$$\sigma(z)=\frac{1}{1+e^{-z}}, \qquad \tanh(z), \qquad \operatorname{ReLU}(z)=\max(0,z)$$

In [ ]:
def sigmoid(z): return 1/(1+np.exp(-z))
def relu(z): return np.maximum(0,z)
def tanh(z): return np.tanh(z)

z_grid = np.linspace(-6,6,400)
plt.figure(figsize=(10,5))
plt.plot(z_grid, sigmoid(z_grid), label='sigmoid')
plt.plot(z_grid, tanh(z_grid), label='tanh')
plt.plot(z_grid, relu(z_grid), label='ReLU')
plt.ylim(-1.5,6)
plt.grid(alpha=.25)
plt.legend()
plt.title('Common activation functions')
plt.xlabel('z')
plt.ylabel('activation')
plt.show()

### Why the activation matters

If every layer were linear, composing layers would still produce a single linear transformation:

$$W_2(W_1x+b_1)+b_2 = (W_2W_1)x + (W_2b_1+b_2)$$

So depth without nonlinearity does **not** buy nonlinear representational power. Activations are what allow stacked layers to bend, fold, clip, and reshape representation space.

## 4. Two-input neuron = a decision boundary

For two inputs:

$$z = w_1x_1+w_2x_2+b$$

The set of points where the neuron is exactly neutral, $z=0$, satisfies:

$$w_1x_1+w_2x_2+b=0$$

That equation is a straight line in 2D. The neuron therefore divides the plane into two half-spaces.

In [ ]:
def plot_boundary(w1, w2, b):
    xs = np.linspace(-4,4,200)
    ys = np.linspace(-4,4,200)
    X1, X2 = np.meshgrid(xs, ys)
    Z = w1*X1 + w2*X2 + b
    plt.figure(figsize=(7,6))
    plt.contourf(X1, X2, Z>0, levels=1, alpha=.3)
    plt.contour(X1, X2, Z, levels=[0], linewidths=2)
    plt.quiver(0,0,w1,w2,angles='xy',scale_units='xy',scale=1)
    plt.xlim(-4,4); plt.ylim(-4,4)
    plt.axhline(0, linewidth=.8); plt.axvline(0, linewidth=.8)
    plt.xlabel('x1'); plt.ylabel('x2')
    plt.title(f'{w1:.1f}x1 + {w2:.1f}x2 + {b:.1f} = 0')
    plt.grid(alpha=.2)
    plt.show()

widgets.interact(
    plot_boundary,
    w1=widgets.FloatSlider(value=1,min=-3,max=3,step=.1),
    w2=widgets.FloatSlider(value=1,min=-3,max=3,step=.1),
    b=widgets.FloatSlider(value=0,min=-4,max=4,step=.1)
);

### Geometric meaning of $\mathbf{w}$ and $b$

Let $\mathbf{w}=(w_1,w_2)$. Then:

- $\mathbf{w}$ points **perpendicular** to the decision boundary.
- changing the direction of $\mathbf{w}$ **rotates** the boundary.
- changing $\|\mathbf{w}\|$ changes how rapidly $z$ changes across space.
- changing $b$ **translates** the boundary.

> This is the first major bridge between **neural-network parameters** and **geometry**.

## 5. Final mental model

A single artificial neuron performs this pipeline:

$$\text{inputs} \rightarrow \text{weighted sum} \rightarrow \text{bias shift} \rightarrow \text{activation} \rightarrow \text{output}$$

Or algebraically:

$$\mathbf{x} \rightarrow z=\mathbf{w}^{\top}\mathbf{x}+b \rightarrow a=\phi(z)$$

### Keep these three intuitions

1. **Weights control direction and sensitivity.**
2. **Bias controls translation/threshold.**
3. **Activation controls nonlinear shape.**

Notebook 02 will take this exact primitive and compose many neurons into a feed-forward network.